# Recs 003: Query embed + top‑K retrieval

## Key Goal

Smoke-test end-to-end query embedding plus top-K retrieval to confirm pipeline correctness.

## Decision It Supports

Whether query->embed->retrieve works reliably before running heavier offline evaluations.

## Primary Metrics

Top-K sanity outputs and basic retrieval plausibility checks for representative queries.


# 1) Setup and loading



In [ ]:
from __future__ import annotations

import json
import os
from pathlib import Path

import numpy as np
import pandas as pd


def _repo_root() -> Path:
    here = Path.cwd().resolve()
    for d in [here, *here.parents]:
        if (d / "pyproject.toml").is_file():
            return d
    raise RuntimeError(f"Could not find repo root (no pyproject.toml). cwd={here}")


REPO_ROOT = _repo_root()
ARTIFACT_DIR = REPO_ROOT / "artifacts" / "recs" / "embeddings" / "game_profile" / "default"
NPZ_PATH = ARTIFACT_DIR / "game_profile_embeddings.npz"
INDEX_PATH = ARTIFACT_DIR / "game_profile_embedding_index.parquet"
META_PATH = ARTIFACT_DIR / "game_profile_embedding_meta.json"

for p in (NPZ_PATH, INDEX_PATH, META_PATH):
    if not p.is_file():
        raise FileNotFoundError(f"Run recs_002 first. Missing {p}")

meta = json.loads(META_PATH.read_text(encoding="utf-8"))
TFHUB_URL = meta["model_name"]
EMBED_DIM = int(meta["dim"])
MAX_CHARS = meta.get("max_chars_per_review")

print("TF Hub:", TFHUB_URL)
print("dim:", EMBED_DIM, "n_games:", meta.get("n_games"))

TF Hub: https://tfhub.dev/google/universal-sentence-encoder/4
dim: 512 n_games: 315


In [ ]:
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")

import tensorflow as tf
import tensorflow_hub as hub

for g in tf.config.list_physical_devices("GPU"):
    try:
        tf.config.experimental.set_memory_growth(g, True)
    except Exception:
        pass

embed_fn = hub.load(TFHUB_URL)

2026-04-21 09:30:06.342008: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1776778206.362791   58152 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1776778206.370277   58152 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1776778206.387270   58152 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776778206.387291   58152 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776778206.387293   58152 computation_placer.cc:177] computation placer alr

# 2) Load embedding index

Run **load-index** (next cell) to load **`X`** and **`idx_df`**. **Section 3** immediately after that lists every `app_id` / `app_name`.

*(Index file = Parquet, not JSON. Model metadata = read **`META_PATH`** with `json.load`.)*

In [ ]:
z = np.load(NPZ_PATH)
X = np.asarray(z["embeddings"], dtype=np.float32)
app_ids_npz = np.asarray(z["app_id"], dtype=np.int64)
z.close()

idx_df = pd.read_parquet(INDEX_PATH)
if len(idx_df) != X.shape[0]:
    raise ValueError("Index rows must match embedding rows")
if not np.array_equal(idx_df["app_id"].to_numpy(), app_ids_npz):
    raise ValueError("app_id column must match npz app_id order")
if X.shape[1] != EMBED_DIM:
    raise ValueError("meta dim does not match embedding matrix")

print("X:", X.shape, "dtype:", X.dtype)

X: (315, 512) dtype: float32


# 3) Game index catalog (`app_id`, `app_name`)

**`idx_df`** — one row per game, same order as **`X`**. Sorted by **`app_name`** for browsing. *(Parquet source: `INDEX_PATH`.)*

In [ ]:
from IPython.display import HTML, display

games_catalog = idx_df[["app_id", "app_name"]].sort_values("app_name", ignore_index=True)
print(f"{len(games_catalog)} games in index")

chunk_size = 10

# Add explanation for what the number is under the game name
explanation_html = (
    "<div style='margin-bottom:8px;'>"
    "<b>Note:</b> The number shown directly under each game name is the game's <b>Steam app_id</b>—a unique identifier used internally on Steam and in this recommender."
    "</div>"
)

def make_row(chunk):
    return "<tr>" + "".join(
        f"<td>{row.app_name}<br><span style='color:gray; font-size:small;'>{row.app_id}</span></td>"
        for _, row in chunk.iterrows()
    ) + "".join("<td></td>" for _ in range(chunk_size - len(chunk))) + "</tr>"

rows_html = "\n".join(
    make_row(games_catalog.iloc[i:i+chunk_size])
    for i in range(0, len(games_catalog), chunk_size)
)

header = "".join([f"<th>Game {i+1}</th>" for i in range(chunk_size)])
html_table = f"""
{explanation_html}
<table border="1" style="border-collapse:collapse">
<thead><tr>{header}</tr></thead>
<tbody>
{rows_html}
</tbody>
</table>
"""
display(HTML(html_table))


315 games in index


Game 1,Game 2,Game 3,Game 4,Game 5,Game 6,Game 7,Game 8,Game 9,Game 10
20XX322110,A Hat in Time253230,A Short Hike1055540,A Way Out1222700,ARK: Survival Evolved346110,ATLAS834910,Age of Empires II (2013)221380,Age of Empires: Definitive Edition1017900,American Truck Simulator270880,Among Us945360
Ancestors Legacy620590,Arma 3107410,Artifact583950,Assassin's Creed Odyssey812140,Assassin's Creed Origins582160,Avorion445220,Axiom Verge332200,BATTALION 1944489940,BATTLETECH637090,BERSERK and the Band of the Hawk502280
Baba Is You736260,Banished242920,Batman: Arkham Asylum GOTY Edition35140,Battle Royale Trainer772540,BattleBlock Theater238460,BeamNG.drive284160,Beat Saber620980,BioShock Infinite8870,Black Desert Online582660,Black Mesa362890
Blackwake420290,Bless Online681660,Bloons TD 6960090,Bomber Crew537800,Borderlands 3397540,Broforce274190,Budget Cuts400940,CHRONO TRIGGER613830,Call of Duty: Infinite Warfare292730,Call of Duty: WWII476600
Castle Crashers204360,Cave Story+200900,Celeste504230,Cities: Skylines255710,Clicker Heroes 2629910,Cold Waters541210,Conan Exiles440900,Counter-Strike: Source240,Crash Bandicoot™ N. Sane Trilogy731490,Crusader Kings III1158310
Cube World1128000,Cuphead268910,Cyberdimension Neptunia: 4 Goddesses Online632350,DARK SOULS™ III374320,DARK SOULS™: REMASTERED570940,DEATH STRANDING1190460,DOOM379720,DOOM Eternal782330,DRAGON BALL FighterZ678950,DRAGON QUEST HEROES™ II574050
DUSK519860,DYNASTY WARRIORS 9730310,Danganronpa 2: Goodbye Despair413420,Danganronpa: Trigger Happy Havoc413410,Darkest Dungeon®262060,Darksiders III606280,Day of Infamy447820,Dead Cells588650,Dead Rising 4543460,Dead by Daylight381210
Deep Rock Galactic548430,Desolate671510,Detention555220,Deus Ex: The Fall258180,Devil May Cry HD Collection631510,DiRT 4421020,Dishonored205100,Divinity: Original Sin 2435150,Doki Doki Literature Club698780,Don't Escape: 4 Days to Survive611760
Don't Starve219740,Don't Starve Together322330,Down To One334040,Dragon Cliff 龙崖758190,Duck Game312530,Due Process753650,Dungreed753420,Dying Light239140,Eco382310,Enter the Gungeon311690
Euro Truck Simulator 2227300,Europa Universalis IV236850,FAR: Lone Sails609320,FINAL FANTASY XII THE ZODIAC AGE595520,FINAL FANTASY XIV Online39210,FINAL FANTASY XV WINDOWS EDITION637650,FTL: Faster Than Light212680,Factorio427520,Fairy Fencer F Advent Dark Force524580,Fallout 4377160


In [ ]:
# Explore the contents of the loaded npz file (`z` was loaded above and then closed)
# The variable name 'z' is used above as the result of np.load(NPZ_PATH).
# Since the file was closed with z.close(), let's reload it just for exploration.
with np.load(NPZ_PATH) as z_explore:
    print("Keys in npz file:", list(z_explore.keys()))
    for key in z_explore.files:
        arr = z_explore[key]
        print(f"{key}: shape={arr.shape}, dtype={arr.dtype}")


Keys in npz file: ['embeddings', 'app_id']
embeddings: shape=(315, 512), dtype=float32
app_id: shape=(315,), dtype=int64


In [ ]:
# Review the first record from the 'z' npz file
print("Sample app_id:", app_ids_npz[0])
print("Sample embedding:", X[0][:20])

Sample app_id: 70
Sample embedding: [-0.00095004 -0.03672563 -0.03502039  0.01601444  0.00848117 -0.02323521
  0.04695653 -0.02192483  0.08485302  0.00052595  0.09227538  0.02433009
 -0.00966642 -0.01924189 -0.01223528  0.03338224 -0.02947254  0.00785661
  0.00577108 -0.02801239]


# 4) Query retrieval helpers

Game rows are **L2-normalized** in `recs_002`, so **dot product = cosine similarity** after the query vector is normalized the same way.


In [ ]:
MIN_QUERY_CHARS = 20
MIN_QUERY_WORDS = 4


def l2_normalize(v: np.ndarray) -> np.ndarray:
    v = np.asarray(v, dtype=np.float32).ravel()
    nrm = np.linalg.norm(v)
    if nrm <= 1e-12:
        return v
    return (v / nrm).astype(np.float32)


def query_quality(text: str) -> dict[str, object]:
    t = (text or "").strip()
    words = [w for w in t.split() if w]
    short_chars = len(t) < MIN_QUERY_CHARS
    short_words = len(words) < MIN_QUERY_WORDS
    too_short = short_chars or short_words
    confidence = "low" if too_short else "normal"
    reason = None
    if too_short:
        reason = (
            f"Query is short/vague ({len(t)} chars, {len(words)} words). "
            f"Consider a follow-up or richer preference text."
        )
    return {
        "clean_text": t,
        "n_chars": len(t),
        "n_words": len(words),
        "too_short": too_short,
        "confidence": confidence,
        "reason": reason,
    }


def embed_query(text: str, verbose: bool = True) -> np.ndarray:
    info = query_quality(text)
    t = str(info["clean_text"])
    if MAX_CHARS is not None:
        t = t[: int(MAX_CHARS)]
    if verbose and bool(info["too_short"]):
        print(f"[query_quality:{info['confidence']}] {info['reason']}")
    out = embed_fn([t])
    return l2_normalize(out)


def top_k_games(query: str, k: int = 10, exclude_app_ids: set[int] | None = None) -> pd.DataFrame:
    info = query_quality(query)
    q = embed_query(str(info["clean_text"]), verbose=False)
    scores = X @ q
    n = len(scores)
    k_eff = min(k, n)
    if exclude_app_ids:
        order_all = np.argsort(-scores)
        picked: list[int] = []
        for i in order_all:
            if int(idx_df["app_id"].iloc[int(i)]) in exclude_app_ids:
                continue
            picked.append(int(i))
            if len(picked) >= k_eff:
                break
        order = np.asarray(picked, dtype=np.int64)
    else:
        part = np.argpartition(-scores, k_eff - 1)[:k_eff]
        order = part[np.argsort(-scores[part])]
    rows = idx_df.iloc[order].copy()
    rows["score"] = scores[order]
    rows["query_confidence"] = str(info["confidence"])
    return rows.reset_index(drop=True)


# 5) Structured preference retrieval (`extract_preferences` → `build_embedding_input`)

This is the MVP product path:

- Start from user draft text (`raw_query`)
- Extract a structured preference profile
- Build one deterministic normalized string for embedding
- Reuse `top_k_games(...)` for retrieval

`raw draft → embed` remains a baseline/ablation; structured is the intended default.

In [ ]:
from steam_review_ml.recommender import build_embedding_input, extract_preferences

raw_query = "I loved the serious tactical sandbox feel and his bald head, but I dislike grindy, overly long progression."
raw_query = "Agent 47 is a bald assassin that I love, but I hated the game"
raw_query = "Hitman is a historic franchise. I hated the second game, but I love the rest of the series featuring that bald assassin"

prefs = extract_preferences(raw_query)
structured_query = build_embedding_input(prefs, raw_query)

print("Raw query:\n", raw_query)
print("\nExtracted profile:\n", json.dumps(prefs, indent=2))
print("\nStructured query text:\n", structured_query)

print("\nTop-K from structured query:")
display(top_k_games(structured_query, k=10))


Raw query:
 Hitman is a historic franchise. I hated the second game, but I love the rest of the series featuring that bald assassin

Extracted profile:
 {
  "likes": [
    "the rest of the series featuring that bald assassin"
  ],
  "dislikes": [
    "the second game"
  ],
  "must_have": [],
  "avoid": [
    "the second game"
  ],
  "style_tone": [],
  "constraints": [],
  "confidence": "normal",
  "raw_query": "Hitman is a historic franchise. I hated the second game, but I love the rest of the series featuring that bald assassin"
}

Structured query text:
 WANTS: the rest of the series featuring that bald assassin. AVOIDS: the second game. MUST_HAVE: none. DISLIKES: the second game. STYLE_TONE: none. CONSTRAINTS: none. CONFIDENCE: normal. OPTIONAL_CONTEXT: i love the rest of the series featuring that bald assassin.

Top-K from structured query:


,app_id,app_name,score,query_confidence
0,863550,HITMAN™ 2,0.459189,normal
1,355790,Styx: Shards of Darkness,0.453499,normal
2,205100,Dishonored,0.444448,normal
3,760060,Mutant Year Zero: Road to Eden,0.431002,normal
4,812140,Assassin's Creed Odyssey,0.428624,normal
5,413420,Danganronpa 2: Goodbye Despair,0.427624,normal
6,214560,Mark of the Ninja,0.426647,normal
7,606280,Darksiders III,0.425520,normal
8,582160,Assassin's Creed Origins,0.420406,normal
9,427290,Vampyr,0.420116,normal


# 6) Example query runs (edit freely)

Treat scores as **cosine similarity** in \[-1, 1\] (typically positive for USE on similar prose).

In [ ]:
DEMO_QUERIES = [
    "I love planes",
    "I love the realism",
    "Absolute seriousness at the center, absurdity everywhere else.",
    "Story-rich single-player RPG with choices and replay value",
    "Fast competitive FPS, skill-based, esports vibe",
    "Relaxing cozy farming or life sim, low stress",
    "Worst game ever",
    "Immersive open-world RPG with deep lore and exploration",
    "Thoughtful indie game with strong narrative",
    "Funny and lighthearted co-op party game",
    "Deep and complex tactical RPG",
    "Easy-to-learn, hard-to-master sports game",
    "Intense first-person shooter with high skill ceiling",
    "Beautiful and serene puzzle game",
    "Very very nuanced system builder",
    
]

for q in DEMO_QUERIES:
    print("\n===", q[:100], "..." if len(q) > 100 else "", "===")
    display(top_k_games(q, k=8))


=== I love planes  ===


,app_id,app_name,score,query_confidence
0,269950,X-Plane 11,0.368106,low
1,537800,Bomber Crew,0.273216,low
2,227300,Euro Truck Simulator 2,0.256583,low
3,270880,American Truck Simulator,0.235718,low
4,47890,The Sims(TM) 3,0.230478,low
5,1291340,Townscaper,0.226114,low
6,1118200,People Playground,0.221852,low
7,585420,Trailmakers,0.212175,low



=== I love the realism  ===


,app_id,app_name,score,query_confidence
0,1118200,People Playground,0.385585,low
1,284160,BeamNG.drive,0.373515,low
2,107410,Arma 3,0.367447,low
3,227300,Euro Truck Simulator 2,0.367230,low
4,47890,The Sims(TM) 3,0.338189,low
5,270880,American Truck Simulator,0.337556,low
6,541210,Cold Waters,0.334646,low
7,447820,Day of Infamy,0.332033,low



=== Absolute seriousness at the center, absurdity everywhere else.  ===


...

,app_id,app_name,score,query_confidence
0,581320,Insurgency: Sandstorm,0.471310,normal
1,1229490,ULTRAKILL,0.460189,normal
2,359550,Tom Clancy's Rainbow Six Siege,0.439983,normal
3,240,Counter-Strike: Source,0.433158,normal
4,447820,Day of Infamy,0.426906,normal
5,519860,DUSK,0.420370,normal
6,311690,Enter the Gungeon,0.416771,normal
7,312660,Sniper Elite 4,0.406044,normal



=== Beautiful and serene puzzle game  ===


,app_id,app_name,score,query_confidence
0,288160,The Room,0.588488,normal
1,736260,Baba Is You,0.582770,normal
2,1289310,Helltaker,0.569212,normal
3,683320,GRIS,0.561806,normal
4,425580,The Room Two,0.560291,normal
5,1055540,A Short Hike,0.523090,normal
6,609320,FAR: Lone Sails,0.522342,normal
7,620,Portal 2,0.517393,normal



=== Very very nuanced system builder  ===


,app_id,app_name,score,query_confidence
0,621060,PC Building Simulator,0.269875,normal
1,352550,Urban Empire,0.266505,normal
2,255710,Cities: Skylines,0.265205,normal
3,242920,Banished,0.239949,normal
4,464920,Surviving Mars,0.229708,normal
5,690830,Foundation,0.225676,normal
6,281990,Stellaris,0.224433,normal
7,236850,Europa Universalis IV,0.223197,normal


In [ ]:
MY_CIV_6_REVIEW = [
    "Sean Bean's voice is great.",
    "Sean Bean's voice is great when you learn a technology",
    "Systematic city builder that stands the test of time",
    "I love working my way through the technology tree",
    "I love the combat system",
    "They added district building",
    "I love the wonders",
    "I get immense satisfaction from building natural wonders",
    "The late-game is a little bit of a grind",
]

for q in MY_CIV_6_REVIEW:
    print("\n===", q[:60], "..." if len(q) > 60 else "", "===")
    display(top_k_games(q, k=8))


=== Sean Bean's voice is great.  ===


,app_id,app_name,score,query_confidence
0,238460,BattleBlock Theater,0.332691,normal
1,213670,South Park™: The Stick of Truth™,0.313439,normal
2,420,Half-Life 2: Episode Two,0.294581,normal
3,274190,Broforce,0.291416,normal
4,362890,Black Mesa,0.291104,normal
5,508440,Totally Accurate Battle Simulator,0.290430,normal
6,548430,Deep Rock Galactic,0.286934,normal
7,221380,Age of Empires II (2013),0.282201,normal



=== Sean Bean's voice is great when you learn a technology  ===


,app_id,app_name,score,query_confidence
0,289070,Sid Meier's Civilization VI,0.329863,normal
1,221380,Age of Empires II (2013),0.289604,normal
2,8930,Sid Meier's Civilization V,0.287088,normal
3,1017900,Age of Empires: Definitive Edition,0.286556,normal
4,508440,Totally Accurate Battle Simulator,0.283570,normal
5,537800,Bomber Crew,0.276864,normal
6,541210,Cold Waters,0.274770,normal
7,238460,BattleBlock Theater,0.271084,normal



=== Systematic city builder that stands the test of time  ===


...

,app_id,app_name,score,query_confidence
0,1118200,People Playground,0.400877,low
1,688130,Pogostuck: Rage With Your Friends,0.388283,low
2,1291340,Townscaper,0.372762,low
3,238460,BattleBlock Theater,0.370206,low
4,240720,Getting Over It with Bennett Foddy,0.367499,low
5,1055540,A Short Hike,0.364599,low
6,47890,The Sims(TM) 3,0.359008,low
7,1089980,The Henry Stickmin Collection,0.351157,low



=== I get immense satisfaction from building natural wonders  ===


,app_id,app_name,score,query_confidence
0,1291340,Townscaper,0.388043,normal
1,255710,Cities: Skylines,0.337834,normal
2,526870,Satisfactory,0.329912,normal
3,242920,Banished,0.323782,normal
4,105600,Terraria,0.318560,normal
5,688130,Pogostuck: Rage With Your Friends,0.318264,normal
6,427520,Factorio,0.317450,normal
7,323190,Frostpunk,0.315864,normal



=== The late-game is a little bit of a grind  ===


,app_id,app_name,score,query_confidence
0,758190,Dragon Cliff 龙崖,0.462153,normal
1,677120,Heroes of Hammerwatch,0.419984,normal
2,646570,Slay the Spire,0.409709,normal
3,212680,FTL: Faster Than Light,0.408803,normal
4,688130,Pogostuck: Rage With Your Friends,0.405695,normal
5,583950,Artifact,0.403037,normal
6,753420,Dungreed,0.402628,normal
7,8930,Sid Meier's Civilization V,0.397890,normal


# 7) Lowest-similarity inspection

Cosine-low does not mean "opposite taste" in a clean semantic sense. It often means:

- different vocabulary/style from the model's training geometry
- sparse/noisy signal
- query nuance not captured by one vector direction

So "lowest" is usually not a reliable anti-recommendation set.

In [ ]:
DEMO_QUERIES = [
    "I love how long it took to complete the game.",
    "I hate how long it took to complete the game.",
    "I love the realism",
    "Absolute seriousness at the center, absurdity everywhere else.",
    "Story-rich single-player RPG with choices and replay value",
    "Fast competitive FPS, skill-based, esports vibe",
    "Relaxing cozy farming or life sim, low stress",
    "Intense first-person shooter with high skill ceiling",
    "Beautiful and serene puzzle game",
    "Very very nuanced system builder",
]

try:
    # Try to get all games (requires top_k_games to allow k larger than or == len(game_profiles))
    test_results = top_k_games(DEMO_QUERIES[0], k=99999)
    total_games = len(test_results)
    if total_games < 100:
        print("Warning: top_k_games may be capped; lowest-score ranking may be incomplete!")
except Exception as e:
    print("Warning: Could not determine number of games automatically; using k=1000 fallback.")
    total_games = 1000

for q in DEMO_QUERIES:
    print("\n=== LOWEST SCORED for:", q[:100], "..." if len(q) > 100 else "", "===")
    # Get ALL games (as many as possible), then sort to show lowest scores.
    all_results = top_k_games(q, k=total_games)
    if len(all_results) < total_games:
        print(f"Note: Only {len(all_results)} games retrieved -- 'lowest' is within these.")
    lowest = all_results.nsmallest(8, "score")
    display(lowest)

# Explanation:
# - We attempt to retrieve as many games as possible for each query.
# - Sorting by score ascending gives the games least similar to the query (truly lowest score).
# - If top_k_games is capped, you may not get true global lowest -- check a warning above.


=== LOWEST SCORED for: I love how long it took to complete the game.  ===


,app_id,app_name,score,query_confidence
314,875210,三国群英传8 Heroes of the Three Kingdoms 8,0.065087,normal
313,269950,X-Plane 11,0.229984,normal
312,421020,DiRT 4,0.251985,normal
311,292730,Call of Duty: Infinite Warfare,0.295187,normal
310,541210,Cold Waters,0.302650,normal
309,673880,"Warhammer 40,000: Mechanicus",0.306982,normal
308,485510,Nioh: Complete Edition,0.307948,normal
307,606280,Darksiders III,0.318643,normal



=== LOWEST SCORED for: I hate how long it took to complete the game.  ===


,app_id,app_name,score,query_confidence
314,875210,三国群英传8 Heroes of the Three Kingdoms 8,0.092078,normal
313,269950,X-Plane 11,0.231473,normal
312,421020,DiRT 4,0.260296,normal
311,541210,Cold Waters,0.301368,normal
310,519860,DUSK,0.309882,normal
309,238320,Outlast,0.311945,normal
308,673880,"Warhammer 40,000: Mechanicus",0.314206,normal
307,332200,Axiom Verge,0.326096,normal



=== LOWEST SCORED for: I love the realism  ===


...

,app_id,app_name,score,query_confidence
314,875210,三国群英传8 Heroes of the Three Kingdoms 8,0.086021,normal
313,431960,Wallpaper Engine,0.128366,normal
312,613830,CHRONO TRIGGER,0.131335,normal
311,637670,Secret of Mana,0.144466,normal
310,825630,STEINS;GATE 0,0.145934,normal
309,1291340,Townscaper,0.149963,normal
308,648350,Jurassic World Evolution,0.150123,normal
307,412830,STEINS;GATE,0.150851,normal



=== LOWEST SCORED for: Beautiful and serene puzzle game  ===


,app_id,app_name,score,query_confidence
314,875210,三国群英传8 Heroes of the Three Kingdoms 8,0.157674,normal
313,841370,NBA 2K19,0.197623,normal
312,577800,NBA 2K18,0.201070,normal
311,269950,X-Plane 11,0.201350,normal
310,489830,The Elder Scrolls V: Skyrim Special Edition,0.206888,normal
309,817130,WWE 2K19,0.209606,normal
308,292730,Call of Duty: Infinite Warfare,0.209749,normal
307,619290,Out of the Park Baseball 19,0.210628,normal



=== LOWEST SCORED for: Very very nuanced system builder  ===


,app_id,app_name,score,query_confidence
314,337340,Finding Paradise,0.047867,normal
313,825630,STEINS;GATE 0,0.058640,normal
312,551730,Toukiden 2,0.059438,normal
311,412830,STEINS;GATE,0.059993,normal
310,543460,Dead Rising 4,0.061182,normal
309,238320,Outlast,0.061786,normal
308,413420,Danganronpa 2: Goodbye Despair,0.063174,normal
307,502280,BERSERK and the Band of the Hawk,0.063656,normal


# 8) Negative-penalty ranking — method and example

Using **lowest cosine scores** as "not recommended" is often misleading. A low score can mean "different language/genre" rather than truly "opposite preference."

A better text-only approach (no tags/hours metadata required):

- Build a **positive query** from what the user wants.
- Build a **negative query** from what they want to avoid (e.g., grindy, too long, repetitive).
- Rank with a penalty:

`score = sim(q_pos, game) - lambda_neg * sim(q_neg, game)`

- Define/edit:
  - `POS_Q` = what the user wants
  - `NEG_Q` = what the user wants to avoid
  - `lambda_neg` = penalty weight
- Execute:
  - `rank_with_negative_penalty(POS_Q, NEG_Q, k=12, lambda_neg=0.6)`

Quick tuning notes:

- Start with `lambda_neg = 0.5`
- If bad matches still appear, increase toward `0.7–1.0`
- If results get too narrow/harsh, lower toward `0.2–0.4`

This keeps retrieval driven by positive intent while explicitly pushing down games that semantically match negative intent.

In [ ]:
def rank_with_negative_penalty(
    positive_query: str,
    negative_query: str,
    k: int = 10,
    lambda_neg: float = 0.5,
) -> pd.DataFrame:
    """Text-only ranking with explicit negative-intent penalty.

    score = sim(q_pos, game) - lambda_neg * sim(q_neg, game)
    where sim is cosine (dot product on normalized vectors).
    """
    q_pos = embed_query(positive_query)
    q_neg = embed_query(negative_query)

    pos_scores = X @ q_pos
    neg_scores = X @ q_neg
    final_scores = pos_scores - float(lambda_neg) * neg_scores

    n = len(final_scores)
    k_eff = min(k, n)
    part = np.argpartition(-final_scores, k_eff - 1)[:k_eff]
    order = part[np.argsort(-final_scores[part])]

    out = idx_df.iloc[order].copy()
    out["score_final"] = final_scores[order]
    out["score_pos"] = pos_scores[order]
    out["score_neg"] = neg_scores[order]
    out["lambda_neg"] = float(lambda_neg)
    return out.reset_index(drop=True)


# Example: serious tactical sandbox wanted; grind/bloat disliked.
POS_Q = "Serious tactical sandbox gameplay with creative problem solving and strong stealth systems"
NEG_Q = "Grindy repetitive gameplay, bloated pacing, too long, filler content"

display(rank_with_negative_penalty(POS_Q, NEG_Q, k=12, lambda_neg=0.6))

,app_id,app_name,score_final,score_pos,score_neg,lambda_neg
0,214560,Mark of the Ninja,0.355113,0.556659,0.335911,0.6
1,205100,Dishonored,0.294525,0.515380,0.368092,0.6
2,107410,Arma 3,0.292383,0.478213,0.309718,0.6
3,355790,Styx: Shards of Darkness,0.285992,0.511835,0.376405,0.6
4,359550,Tom Clancy's Rainbow Six Siege,0.283880,0.473434,0.315923,0.6
5,541210,Cold Waters,0.266173,0.447123,0.301583,0.6
6,773951,Freeman: Guerrilla Warfare,0.265553,0.470464,0.341518,0.6
7,460930,Tom Clancy's Ghost Recon® Wildlands,0.261930,0.475440,0.355850,0.6
8,236510,Takedown: Red Sabre,0.261464,0.455582,0.323530,0.6
9,4000,Garry's Mod,0.259569,0.438852,0.298805,0.6


In [ ]:
# Example: serious tactical sandbox wanted; grind/bloat disliked.
POS_Q = "Silly, fun, and lighthearted game with a focus on exploration and discovery"
NEG_Q = "Grindy repetitive gameplay, bloated pacing, too long, filler content"

display(rank_with_negative_penalty(POS_Q, NEG_Q, k=12, lambda_neg=0.6))

,app_id,app_name,score_final,score_pos,score_neg,lambda_neg
0,1055540,A Short Hike,0.282893,0.519766,0.394787,0.6
1,569860,Thimbleweed Park,0.263980,0.480820,0.361399,0.6
2,212680,FTL: Faster Than Light,0.261198,0.504558,0.405601,0.6
3,477160,Human: Fall Flat,0.259341,0.467890,0.347583,0.6
4,1240210,There Is No Game: Wrong Dimension,0.256817,0.490504,0.389478,0.6
5,620,Portal 2,0.252277,0.455643,0.338943,0.6
6,206190,Gunpoint,0.239611,0.498433,0.431371,0.6
7,743450,Monster Prom,0.239576,0.431832,0.320426,0.6
8,105600,Terraria,0.238327,0.472674,0.390578,0.6
9,420290,Blackwake,0.237639,0.434861,0.328704,0.6


# 9) Ablation: structured vs raw vs negative penalty

Same user **draft** → compare up to **three** rankings:

| Path | Mechanism |
|------|-----------|
| **Structured (v1 default)** | One string: `build_embedding_input(extract_preferences(draft), draft)` → embed → cosine vs games |
| **Raw (baseline)** | Embed the **draft** verbatim |
| **Negative penalty** | Uses **`rank_with_negative_penalty`** from **§8**: `score = sim(pos) − λ·sim(neg)` (two embeds). **Pos** / **neg** strings are built from the extracted profile (wants/style/constraints vs dislikes/avoid). If the profile has **no** dislikes and **no** avoid lines, this column is **skipped** (nothing meaningful to embed as `neg`). |

Tune **`LAMBDA_ABLATION`** below with your eval set; it interacts with how strong the neg string is.

### Toward Precision@K (and related metrics)

**Precision@K** needs a **definition of “relevant”** per query (or per user–query). You cannot compute it from similarities alone. A minimal way to start:

1. **Fixed draft set** — 20–100 representative queries (like your ablation drafts plus edge cases).
2. **Relevance labels** — for each draft, mark which `app_id`s in the catalog are **relevant** (binary), at least within top‑M candidates if labeling the full catalog is too heavy.
3. **Score** — for each method (**structured**, **raw**, **negative penalty**), take the top‑K `app_id`s; **Precision@K** = (labeled relevant items in top‑K) / K. Average over drafts.

Related: **Recall@K** (needs enough labels per query), **MAP@K**, **NDCG@K** if you have graded relevance. **Pairwise overlap@K** (printed below) is **not** precision—it only measures agreement between rankers.

Semi-automated options later: weak labels from “user played > N hours,” wishlist, or thumbs; or LLM-assisted labeling with human audit on a subset.

In [ ]:
from IPython.display import HTML, display

from steam_review_ml.recommender import build_embedding_input, extract_preferences

# Strength of neg similarity in §8 formula: score = sim_pos - LAMBDA_ABLATION * sim_neg
LAMBDA_ABLATION = 0.6


def _preview(text: str, n: int = 480) -> str:
    t = (text or "").strip()
    return t if len(t) <= n else t[: n].rstrip() + "…"


def _csv(xs: object) -> str:
    items = [str(x).strip() for x in (xs or []) if str(x).strip()]
    return ", ".join(items)


def prefs_to_pos_neg_strings(prefs: dict[str, object], structured_fallback: str) -> tuple[str, str, bool]:
    """Build pos/neg query text for rank_with_negative_penalty from extract_preferences output."""
    likes = _csv(prefs.get("likes"))
    must_have = _csv(prefs.get("must_have"))
    style = _csv(prefs.get("style_tone"))
    constraints = _csv(prefs.get("constraints"))
    pos_bits = []
    if likes:
        pos_bits.append(f"WANTS: {likes}")
    if must_have:
        pos_bits.append(f"MUST_HAVE: {must_have}")
    if style:
        pos_bits.append(f"STYLE: {style}")
    if constraints:
        pos_bits.append(f"CONSTRAINTS: {constraints}")
    pos_q = ". ".join(pos_bits) if pos_bits else structured_fallback

    dislikes = _csv(prefs.get("dislikes"))
    avoid = _csv(prefs.get("avoid"))
    neg_bits = []
    if dislikes:
        neg_bits.append(f"DISLIKES: {dislikes}")
    if avoid:
        neg_bits.append(f"AVOIDS: {avoid}")
    neg_q = ". ".join(neg_bits)
    use_penalty = bool(neg_q)
    return pos_q, neg_q, use_penalty


def show_ablation_tables(
    structured_df: pd.DataFrame,
    raw_df: pd.DataFrame,
    penalty_df: pd.DataFrame | None,
    *,
    k_show: int,
) -> None:
    left = structured_df.head(k_show)[["app_id", "app_name", "score"]].copy()
    left.columns = ["app_id", "app_name", "score (structured)"]
    mid = raw_df.head(k_show)[["app_id", "app_name", "score"]].copy()
    mid.columns = ["app_id", "app_name", "score (raw)"]
    lhtml = left.to_html(index=False, float_format=lambda x: f"{x:.4f}")
    mhtml = mid.to_html(index=False, float_format=lambda x: f"{x:.4f}")
    if penalty_df is not None:
        right = penalty_df.head(k_show)[["app_id", "app_name", "score_final"]].copy()
        right.columns = ["app_id", "app_name", "score (pos−λ·neg)"]
        rhtml = right.to_html(index=False, float_format=lambda x: f"{x:.4f}")
        third = (
            f'<div style="flex:1;min-width:300px;"><h4 style="margin-top:0">Negative penalty (§8)</h4>'
            f"<p style=\"font-size:small;margin:0 0 8px 0\">λ = {LAMBDA_ABLATION}</p>{rhtml}</div>"
        )
    else:
        third = (
            '<div style="flex:1;min-width:300px;"><h4 style="margin-top:0">Negative penalty</h4>'
            "<p style=\"font-size:small\">Skipped — no DISLIKES/AVOIDS in profile.</p></div>"
        )
    html = (
        '<div style="display:flex;gap:20px;align-items:flex-start;flex-wrap:wrap;">'
        '<div style="flex:1;min-width:300px;"><h4 style="margin-top:0">Structured (single embed)</h4>'
        + lhtml
        + '</div><div style="flex:1;min-width:300px;"><h4 style="margin-top:0">Raw draft</h4>'
        + mhtml
        + "</div>"
        + third
        + "</div>"
    )
    display(HTML(html))


def ablation_compare(raw_draft: str, k: int = 10) -> None:
    prefs = extract_preferences(raw_draft)
    structured_text = build_embedding_input(prefs, raw_draft)
    pos_q, neg_q, use_penalty = prefs_to_pos_neg_strings(prefs, structured_text)

    df_s = top_k_games(structured_text, k=k)
    df_r = top_k_games(raw_draft, k=k)
    df_p: pd.DataFrame | None
    if use_penalty:
        df_p = rank_with_negative_penalty(pos_q, neg_q, k=k, lambda_neg=LAMBDA_ABLATION)
    else:
        df_p = None

    print("Draft:\n", raw_draft)
    print("\nStructured embedding input:\n", _preview(structured_text))
    if use_penalty:
        print("\nNegative-penalty pos string:\n", _preview(pos_q))
        print("\nNegative-penalty neg string:\n", _preview(neg_q))
    show_ablation_tables(df_s, df_r, df_p, k_show=k)

    s_ids = set(df_s["app_id"].iloc[:k].tolist())
    r_ids = set(df_r["app_id"].iloc[:k].tolist())
    print(f"Top-{k} overlap structured ∩ raw: {len(s_ids & r_ids)} / {k}")
    if df_p is not None:
        p_ids = set(df_p["app_id"].iloc[:k].tolist())
        print(f"Top-{k} overlap structured ∩ penalty: {len(s_ids & p_ids)} / {k}")
        print(f"Top-{k} overlap raw ∩ penalty: {len(r_ids & p_ids)} / {k}")
    print()


# Requires §4 (`top_k_games`) and §8 (`rank_with_negative_penalty`) run above.
ABLATION_DRAFTS = [
    "I loved the serious tactical sandbox feel, but I dislike grindy, overly long progression.",
    "Agent 47 is a bald assassin that I love, but I hated the game",
    "Hitman is a historic franchise. I hated the second game, but I love the rest of the series featuring that bald assassin",
]

K_ABLATION = 10
for i, draft in enumerate(ABLATION_DRAFTS):
    print("=" * 72)
    print(f"Ablation {i + 1} / {len(ABLATION_DRAFTS)}")
    ablation_compare(draft, k=K_ABLATION)


Ablation 1 / 3
Draft:
 I loved the serious tactical sandbox feel, but I dislike grindy, overly long progression.

Structured embedding input:
 WANTS: the serious tactical sandbox feel. AVOIDS: overly long or bloated progression, grindy or repetitive progression loops. MUST_HAVE: none. DISLIKES: grindy, overly long progression. STYLE_TONE: serious, tactical, sandbox. CONSTRAINTS: shorter pacing preferred, low grind preferred. CONFIDENCE: normal. OPTIONAL_CONTEXT: i dislike grindy, overly long progression.

Negative-penalty pos string:
 WANTS: the serious tactical sandbox feel. STYLE: serious, tactical, sandbox. CONSTRAINTS: shorter pacing preferred, low grind preferred

Negative-penalty neg string:
 DISLIKES: grindy, overly long progression. AVOIDS: overly long or bloated progression, grindy or repetitive progression loops


app_id,app_name,score (structured)
758190,Dragon Cliff 龙崖,0.5247
857980,Void Bastards,0.4858
393520,Iconoclasts,0.4850
677120,Heroes of Hammerwatch,0.4838
753420,Dungreed,0.4814
629910,Clicker Heroes 2,0.4752
527230,For The King,0.4685
858210,Nova Drift,0.4671
760060,Mutant Year Zero: Road to Eden,0.4667
588650,Dead Cells,0.4646


Top-10 overlap structured ∩ raw: 4 / 10
Top-10 overlap structured ∩ penalty: 0 / 10
Top-10 overlap raw ∩ penalty: 0 / 10

Ablation 2 / 3
Draft:
 Agent 47 is a bald assassin that I love, but I hated the game

Structured embedding input:
 WANTS: none. AVOIDS: the game. MUST_HAVE: none. DISLIKES: the game. STYLE_TONE: none. CONSTRAINTS: none. CONFIDENCE: normal. OPTIONAL_CONTEXT: i hated the game.

Negative-penalty pos string:
 WANTS: none. AVOIDS: the game. MUST_HAVE: none. DISLIKES: the game. STYLE_TONE: none. CONSTRAINTS: none. CONFIDENCE: normal. OPTIONAL_CONTEXT: i hated the game.

Negative-penalty neg string:
 DISLIKES: the game. AVOIDS: the game


app_id,app_name,score (structured)
1180380,Stay Out,0.4389
1240210,There Is No Game: Wrong Dimension,0.4386
960090,Bloons TD 6,0.4385
626690,Sword Art Online: Fatal Bullet,0.4330
704850,Thief Simulator,0.4274
428690,Youtubers Life,0.4232
352550,Urban Empire,0.4225
420530,OneShot,0.4222
760060,Mutant Year Zero: Road to Eden,0.4213
527230,For The King,0.4184


Top-10 overlap structured ∩ raw: 0 / 10
Top-10 overlap structured ∩ penalty: 2 / 10
Top-10 overlap raw ∩ penalty: 0 / 10

Ablation 3 / 3
Draft:
 Hitman is a historic franchise. I hated the second game, but I love the rest of the series featuring that bald assassin

Structured embedding input:
 WANTS: the rest of the series featuring that bald assassin. AVOIDS: the second game. MUST_HAVE: none. DISLIKES: the second game. STYLE_TONE: none. CONSTRAINTS: none. CONFIDENCE: normal. OPTIONAL_CONTEXT: i love the rest of the series featuring that bald assassin.

Negative-penalty pos string:
 WANTS: the rest of the series featuring that bald assassin

Negative-penalty neg string:
 DISLIKES: the second game. AVOIDS: the second game


app_id,app_name,score (structured)
863550,HITMAN™ 2,0.4592
355790,Styx: Shards of Darkness,0.4535
205100,Dishonored,0.4444
760060,Mutant Year Zero: Road to Eden,0.4310
812140,Assassin's Creed Odyssey,0.4286
413420,Danganronpa 2: Goodbye Despair,0.4276
214560,Mark of the Ninja,0.4266
606280,Darksiders III,0.4255
582160,Assassin's Creed Origins,0.4204
427290,Vampyr,0.4201


Top-10 overlap structured ∩ raw: 6 / 10
Top-10 overlap structured ∩ penalty: 2 / 10
Top-10 overlap raw ∩ penalty: 2 / 10

